# 血液抹片細胞自動辨識與計數系統
> 影像處理 114-2 ｜ 期末專案

**使用技術**：OpenCV · Adaptive Thresholding · Morphological Operations · Watershed · Streamlit

| 區塊 | 功能 |
|------|------|
| §0 Setup | 安裝依賴、載入資料集 |
| §1 Utils | 輔助函式 |
| §2 Preprocess | 前處理 Pipeline |
| §3 Detect | 輪廓偵測 |
| §4 Classify | RBC / WBC / Platelet 分類 |
| §5 Watershed | 重疊細胞切分 |
| §6 Pipeline | 總控函式 |
| §7 Evaluation | 5 張圖固定參數評估 |
| §8 App | Streamlit 互動式 App |

## §0 環境設定

In [ ]:
# §0 安裝依賴套件
!pip install -q streamlit

# 下載 cloudflared（免費 tunnel，不需要帳號）
import os
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 下載 BCCD Dataset（若已存在則跳過）
if not os.path.exists('/content/BCCD_Dataset'):
    !git clone https://github.com/Shenggan/BCCD_Dataset.git /content/BCCD_Dataset
else:
    print('BCCD Dataset 已存在')

# 載入常用套件
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xml.etree.ElementTree as ET
import glob
from dataclasses import dataclass

print('✅ 環境設定完成')

## §1 輔助函式

In [ ]:
# §1 輔助函式

def show(title, img):
    '''顯示影像於 notebook（支援灰階與彩色）'''
    plt.figure(figsize=(6, 4))
    plt.title(title)
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


def show_stages(stages_dict, stage_order, stage_labels):
    '''並排顯示多個前處理階段'''
    n = len(stage_order)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    for ax, key in zip(axes, stage_order):
        img = stages_dict[key]
        if len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(stage_labels[key])
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def overlay_contours(img, contours_dict):
    '''
    對每個偵測到的細胞畫最小外接圓（而非原始輪廓），視覺上更清晰。
    RBC：紅色，WBC：綠色，Platelet：藍色
    '''
    result = img.copy()
    colors = {'rbc': (0, 0, 255), 'wbc': (0, 255, 0), 'platelet': (255, 0, 0)}
    for cell_type, contours in contours_dict.items():
        for cnt in contours:
            (x, y), r = cv2.minEnclosingCircle(cnt)
            cv2.circle(result, (int(x), int(y)), int(r), colors[cell_type], 2)
    return result


def encode_image(img):
    '''將 BGR 影像編碼為 PNG bytes，供下載使用'''
    _, buffer = cv2.imencode('.png', img)
    return buffer.tobytes()


print('✅ Utils 載入完成')

## §2 影像前處理 Pipeline

**步驟**：灰階 → Gaussian Blur → Adaptive Thresholding → Morphological Closing

- **Adaptive Thresholding**：血液抹片光照不均，全域閾值效果差；Adaptive 以局部區塊均值為基準，對不均勻背景更穩健
- **Morphological Closing**：先膨脹再侵蝕，填補細胞內部因染色不均產生的空洞

In [ ]:
# §2 前處理 Pipeline

@dataclass
class PreprocessParams:
    gaussian_ksize: int = 5    # Gaussian kernel 大小（必須為奇數）
    adaptive_block: int = 71   # Adaptive threshold 區塊大小；71 ≈ 1.7× 細胞直徑，
                                # 使局部均值含足夠背景像素，讓 RBC 淡色中央也落於均值以下
    adaptive_c:     int = 2    # 從區塊均值減去的常數
    morph_ksize:    int = 5    # 形態學 kernel 大小
    morph_iter:     int = 1    # 形態學操作迭代次數


def preprocess(img, params):
    '''
    前處理 pipeline：灰階 → CLAHE 增強 → 高斯模糊 → Adaptive Thresholding → Morphological Closing → 輪廓填充
    回傳各階段影像 dict，可用於 App 可視化
    '''
    # 1. 灰階轉換
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. CLAHE 影像增強：局部直方圖均衡化，提升染色不均影像的對比度
    #    clipLimit=2.0 限制對比放大倍率（避免過度增強雜訊），tileGridSize=(8,8) 為局部窗口大小
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # 3. 高斯模糊：抑制高頻雜訊，保留細胞邊緣
    blurred = cv2.GaussianBlur(
        gray, (params.gaussian_ksize, params.gaussian_ksize), 0
    )

    # 4. Adaptive Thresholding：以局部均值為基準處理不均勻光照
    #    block=71 確保局部窗口含足夠背景，使 RBC 淡色中央（pale center）
    #    也落於局部均值以下，偵測到更完整的細胞實心區域（非單純環形膜）
    thresh = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        params.adaptive_block,
        params.adaptive_c
    )

    # 5. Morphological Closing：橢圓形 kernel 填補細胞輪廓上的小缺口
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (params.morph_ksize, params.morph_ksize)
    )
    morphed = cv2.morphologyEx(
        thresh, cv2.MORPH_CLOSE, kernel, iterations=params.morph_iter
    )

    # 6. 輪廓填充：將閉合的環形輪廓填充為實心圓盤
    #    （Flood Fill 在 morph_iter≥2 後角落像素可能變白，導致填充失效，
    #      改用 RETR_EXTERNAL 輪廓的 FILLED 繪製更穩定）
    cnts, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(morphed)
    cv2.drawContours(filled, cnts, -1, 255, cv2.FILLED)
    morphed = filled

    return {
        'original': img,
        'gray':     gray,
        'enhanced': enhanced,   # CLAHE 增強結果（供 RBC Hough 偵測使用）
        'blurred':  blurred,
        'thresh':   thresh,
        'morphed':  morphed,
    }


print('✅ Preprocess 載入完成')

In [ ]:
# §2 Demo：顯示前處理各階段結果
sample_img = cv2.imread('/content/BCCD_Dataset/BCCD/JPEGImages/BloodImage_00001.jpg')

if sample_img is None:
    print('⚠️  請先執行 §0 Setup 以載入資料集')
else:
    stages = preprocess(sample_img, PreprocessParams())
    order = ['original', 'gray', 'enhanced', 'blurred', 'thresh', 'morphed']
    labels = {
        'original': '原圖',
        'gray':     '灰階',
        'enhanced': 'CLAHE 增強',
        'blurred':  'Gaussian 模糊',
        'thresh':   'Adaptive Thresholding',
        'morphed':  'Morphological Closing',
    }
    show_stages(stages, order, labels)

## §3 細胞偵測

從 Morphological Closing 結果找輪廓，計算每個輪廓的：
- **面積** (`area`)：像素數量
- **圓形度** (`circularity`)：`4π·A / P²`，完美圓形 = 1，不規則形狀趨近 0

In [ ]:
# §3 細胞偵測：輪廓分析

def detect_cells(binary, min_area=50):
    '''
    從二值影像找出細胞輪廓，計算幾何特徵
    min_area：過濾背景雜訊用的最小面積閾值（px²）
    '''
    contours, _ = cv2.findContours(
        binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cells = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter < 1:
            continue
        # 圓形度：4πA/P²，裁切到 [0, 1] 避免數值誤差
        circularity = min(4 * np.pi * area / (perimeter ** 2), 1.0)
        cells.append({
            'contour':     cnt,
            'area':        area,
            'perimeter':   perimeter,
            'circularity': circularity,
            'bbox':        cv2.boundingRect(cnt),
        })
    return cells


print('✅ Detect 載入完成')

## §4 細胞偵測：三種獨立策略

每類細胞採用最適合其形態與染色特性的偵測方式，而非統一用 Watershed 輪廓分類：

| 細胞 | 偵測方式 | 核心依據 |
|------|----------|---------|
| WBC（白血球） | HSV 紫色遮罩 → 形態學聚合 → 凸包合併 | Giemsa 染色使細胞核呈藍紫色；多葉核透過大 kernel 合併 |
| RBC（紅血球） | CLAHE 增強 → Hough 圓形偵測 → 雙通道顏色驗證 | RBC 為雙凹圓盤；Hough 天然適合偵測圓形；CLAHE 改善染色不均 |
| Platelet（血小板） | HSV 紫色遮罩（小面積）+ WBC 積極排除 | 血小板小型紫色；需排除 WBC 核碎片避免誤判 |

In [ ]:
# §4 細胞偵測：WBC / RBC / Platelet 各自獨立策略

def make_kernel(size):
    size = int(size)
    if size % 2 == 0:
        size += 1
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))


@dataclass
class DetectionConfig:
    method: str = 'circular'
    rbc_sensitive_param2: float = 31
    rbc_strict_param2: float = 33
    rbc_over_count_guard: int = 20
    platelet_rbc_exclusion_pad: int = -6
    platelet_min_area: float = 70
    platelet_max_area: float = 600
    platelet_min_circularity: float = 0.45


# ── WBC 偵測 ─────────────────────────────────────────────────────────────────

def detect_wbc(bgr):
    '''
    WBC 偵測：HSV 紫色遮罩 → 形態學聚合 → 凸包合併
    WBC 核被 Giemsa 染色呈藍紫色（H:100–170, S:60–255, V:50–210）
    中性球多葉核以大 kernel 膨脹 + closing 合併為整塊；相近區塊距離 < 140px 聚類
    '''
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    purple = cv2.inRange(hsv, np.array([100, 60, 50]), np.array([170, 255, 210]))
    purple = cv2.morphologyEx(purple, cv2.MORPH_OPEN, make_kernel(3), iterations=1)
    merged = cv2.dilate(purple, make_kernel(15), iterations=1)
    merged = cv2.morphologyEx(merged, cv2.MORPH_CLOSE, make_kernel(15), iterations=1)

    img_h, img_w = merged.shape
    contours, _ = cv2.findContours(merged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    raw_candidates = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 500:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        touches = x <= 1 or y <= 1 or x + w >= img_w - 1 or y + h >= img_h - 1
        m = cv2.moments(cnt)
        cx = m['m10'] / m['m00'] if m['m00'] else x + w / 2
        cy = m['m01'] / m['m00'] if m['m00'] else y + h / 2
        raw_candidates.append({
            'contour': cnt, 'area': float(area),
            'bbox': (x, y, w, h), 'touches': touches, 'center': (cx, cy)
        })

    # 距離 < 140px 的區塊聚類為同一 WBC（多葉核聚合）
    visited = [False] * len(raw_candidates)
    clusters = []
    for i in range(len(raw_candidates)):
        if visited[i]:
            continue
        stack, cluster = [i], []
        visited[i] = True
        while stack:
            cur = stack.pop()
            cluster.append(cur)
            x1, y1 = raw_candidates[cur]['center']
            for j in range(len(raw_candidates)):
                if not visited[j]:
                    x2, y2 = raw_candidates[j]['center']
                    if ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5 < 140:
                        visited[j] = True
                        stack.append(j)
        clusters.append(cluster)

    # 每個聚類取凸包
    hulls = []
    for cluster in clusters:
        points = np.vstack([raw_candidates[i]['contour'] for i in cluster])
        hull = cv2.convexHull(points)
        area = cv2.contourArea(hull)
        x, y, w, h = cv2.boundingRect(hull)
        touches = x <= 1 or y <= 1 or x + w >= img_w - 1 or y + h >= img_h - 1
        hulls.append({'hull': hull, 'area': float(area),
                      'bbox': (x, y, w, h), 'touches': touches})

    non_border = [h for h in hulls if not h['touches'] and h['area'] >= 6000]
    border     = [h for h in hulls if     h['touches'] and h['area'] >= 6000]

    if non_border:
        selected, status = max(non_border, key=lambda h: h['area']), 'full_wbc'
    elif border:
        selected, status = max(border,     key=lambda h: h['area']), 'partial_wbc_border'
    elif hulls:
        selected, status = max(hulls,      key=lambda h: h['area']), 'no_confident_wbc'
    else:
        selected, status = None, 'no_candidate'

    return {'purple_mask': purple, 'merged_mask': merged,
            'hulls': hulls, 'selected': selected, 'status': status}


# ── RBC 偵測 ─────────────────────────────────────────────────────────────────

def detect_rbc(bgr, param2=32):
    '''
    RBC 偵測：CLAHE 增強 → Hough 圓形偵測 → 雙通道顏色驗證
    - CLAHE：局部直方圖均衡化，改善染色不均導致的偵測遺漏
    - HoughCircles：天然適合 RBC 雙凹圓盤形狀
      param2=32（由 YOLO 輔助診斷 + GT XML 掃描決定，平均誤差 28.1%）
    - 雙通道驗證：HSV 粉紅色範圍 OR Lab a/b 色度，去除誤檢區域
    - WBC 排除：避免 WBC 核心被錯誤計為 RBC
    '''
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    # CLAHE 增強：提升對比，使淡染 RBC 也能被 Hough 偵測
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    blurred = cv2.medianBlur(enhanced, 5)

    circles = cv2.HoughCircles(
        blurred, cv2.HOUGH_GRADIENT, dp=1.2,
        minDist=35, param1=50, param2=param2,
        minRadius=18, maxRadius=55
    )

    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    h, s, v = cv2.split(hsv)
    L, A, B = cv2.split(lab)

    # WBC 排除區：膨脹紫色區域，避免 WBC 碎片被計為 RBC
    purple = cv2.inRange(hsv, np.array([100, 60, 50]), np.array([170, 255, 210]))
    wbc_exclusion = cv2.dilate(purple, make_kernel(25), iterations=1)

    # 雙通道 RBC 顏色驗證
    mask_hsv = (((h >= 150) | (h <= 10)) & (s >= 8) & (s <= 100) &
                (v >= 100) & (v <= 240)).astype(np.uint8) * 255
    mask_lab = ((A >= 132) & (B >= 118) & (L >= 110) & (L <= 225)).astype(np.uint8) * 255
    rbc_mask = cv2.bitwise_or(mask_hsv, mask_lab)
    rbc_mask[wbc_exclusion > 0] = 0
    rbc_mask = cv2.morphologyEx(rbc_mask, cv2.MORPH_OPEN, make_kernel(3), iterations=1)

    accepted = []
    if circles is not None:
        circles_int = np.round(circles[0]).astype(int)
        img_h, img_w = gray.shape
        for x, y, r in circles_int:
            if x < 0 or y < 0 or x >= img_w or y >= img_h:
                continue
            circle_mask = np.zeros(gray.shape, np.uint8)
            cv2.circle(circle_mask, (x, y), r, 255, -1)
            area = np.sum(circle_mask > 0)
            rbc_ratio = np.sum((rbc_mask > 0) & (circle_mask > 0)) / (area + 1e-6)
            wbc_ratio = np.sum((wbc_exclusion > 0) & (circle_mask > 0)) / (area + 1e-6)
            if wbc_ratio > 0.08 or rbc_ratio < 0.15:
                continue
            accepted.append({'x': int(x), 'y': int(y), 'r': int(r),
                              'rbc_ratio': float(rbc_ratio), 'wbc_ratio': float(wbc_ratio)})

    return {'enhanced': enhanced, 'rbc_mask': rbc_mask,
            'wbc_exclusion': wbc_exclusion, 'circles': accepted}


def detect_rbc_circular_rule(bgr, config=None):
    # HoughCircles 產生圓形候選，再補上 area/circularity 作為分類依據。
    # 候選過多時用固定規則切換到較嚴格門檻，避免單張手動調參。
    if config is None:
        config = DetectionConfig()

    sensitive = detect_rbc(bgr, param2=config.rbc_sensitive_param2)
    if len(sensitive['circles']) > config.rbc_over_count_guard:
        result = detect_rbc(bgr, param2=config.rbc_strict_param2)
    else:
        result = sensitive

    for c in result['circles']:
        c['area'] = float(np.pi * c['r'] * c['r'])
        c['circularity'] = 1.0
        c['bbox'] = (int(c['x'] - c['r']), int(c['y'] - c['r']), int(c['r'] * 2), int(c['r'] * 2))

    result['method'] = 'circular_rule'
    return result


# ── Platelet 偵測 ─────────────────────────────────────────────────────────────

def detect_platelets(bgr, wbc_result):
    '''
    Platelet 偵測：HSV 紫色小物件 + 積極 WBC 排除
    血小板也染紫色，但面積遠小於 WBC（8–250 px²）
    WBC 所在區域膨脹 35px 後完全排除，避免 WBC 核碎片誤判為血小板
    '''
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    raw_mask = cv2.inRange(hsv, np.array([110, 80, 30]), np.array([160, 255, 210]))
    raw_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN, make_kernel(3), iterations=1)

    # WBC 排除遮罩
    exclude = np.zeros(raw_mask.shape, np.uint8)
    if wbc_result['selected'] is not None:
        cv2.drawContours(exclude, [wbc_result['selected']['hull']], -1, 255, -1)
        exclude = cv2.dilate(exclude, make_kernel(35), iterations=1)

    platelet_mask = raw_mask.copy()
    platelet_mask[exclude > 0] = 0
    platelet_mask = cv2.morphologyEx(platelet_mask, cv2.MORPH_OPEN, make_kernel(3), iterations=1)

    contours, _ = cv2.findContours(platelet_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_h, img_w = platelet_mask.shape
    platelets = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (8 <= area <= 250):
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        if x <= 1 or y <= 1 or x + w >= img_w - 1 or y + h >= img_h - 1:
            continue
        aspect = w / h if h else 0
        if aspect < 0.35 or aspect > 2.8:
            continue
        perimeter = cv2.arcLength(cnt, True)
        circ = 4 * np.pi * area / (perimeter * perimeter) if perimeter else 0
        if circ < 0.2:
            continue
        m = cv2.moments(cnt)
        cx = int(m['m10'] / m['m00']) if m['m00'] else x + w // 2
        cy = int(m['m01'] / m['m00']) if m['m00'] else y + h // 2
        platelets.append({'contour': cnt, 'area': float(area),
                           'bbox': (x, y, w, h), 'center': (cx, cy), 'circularity': float(circ)})

    return {'raw_mask': raw_mask, 'exclude_mask': exclude,
            'platelet_mask': platelet_mask, 'platelets': platelets}


def detect_platelets_circular_rule(bgr, wbc_result, rbc_result, config=None):
    # HSV/Lab 淡藍紫色遮罩 → WBC 排除 → RBC 圓形區排除 → 面積/圓形度/長寬比篩選。
    if config is None:
        config = DetectionConfig()

    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    h, s, v = cv2.split(hsv)
    L, A, B = cv2.split(lab)

    platelet_mask = (((h >= 105) & (h <= 130) &
                      (s >= 25)  & (s <= 95)  &
                      (v >= 150) & (v <= 225) &
                      (A >= 130) & (A <= 138) &
                      (B >= 108) & (B <= 116) &
                      (L >= 145) & (L <= 205))).astype(np.uint8) * 255

    platelet_mask = cv2.morphologyEx(platelet_mask, cv2.MORPH_OPEN, make_kernel(3), iterations=1)

    exclude = np.zeros(platelet_mask.shape, np.uint8)
    if wbc_result['selected'] is not None:
        cv2.drawContours(exclude, [wbc_result['selected']['hull']], -1, 255, -1)
        exclude = cv2.dilate(exclude, make_kernel(35), iterations=1)

    for c in rbc_result['circles']:
        radius = max(1, int(c['r'] + config.platelet_rbc_exclusion_pad))
        cv2.circle(exclude, (c['x'], c['y']), radius, 255, -1)

    platelet_mask[exclude > 0] = 0
    platelet_mask = cv2.morphologyEx(platelet_mask, cv2.MORPH_CLOSE, make_kernel(5), iterations=1)

    contours, _ = cv2.findContours(platelet_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_h, img_w = platelet_mask.shape
    platelets = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (config.platelet_min_area <= area <= config.platelet_max_area):
            continue

        x, y, w, h_box = cv2.boundingRect(cnt)
        if x <= 5 or y <= 5 or x + w >= img_w - 5 or y + h_box >= img_h - 5:
            continue

        aspect = w / h_box if h_box else 0
        if aspect < 0.5 or aspect > 2.0:
            continue

        perimeter = cv2.arcLength(cnt, True)
        circularity = 4 * np.pi * area / (perimeter * perimeter) if perimeter else 0
        if circularity < config.platelet_min_circularity:
            continue

        m = cv2.moments(cnt)
        cx = int(m['m10'] / m['m00']) if m['m00'] else x + w // 2
        cy = int(m['m01'] / m['m00']) if m['m00'] else y + h_box // 2
        platelets.append({'contour': cnt, 'area': float(area),
                           'bbox': (x, y, w, h_box), 'center': (cx, cy),
                           'circularity': float(circularity), 'aspect': float(aspect)})

    return {'raw_mask': platelet_mask, 'exclude_mask': exclude,
            'platelet_mask': platelet_mask, 'platelets': platelets,
            'method': 'circular_rule'}


print('✅ Detect (WBC / RBC / Platelet) 載入完成')

## §5 Watershed 重疊細胞切分

**為什麼選 Watershed？**  
血液抹片中 RBC 常相鄰排列，形態學無法分離。Watershed 以 Distance Transform 的峰值為「水源」向外擴散，在相鄰細胞之間自動找到分水嶺邊界。

**步驟**
1. 膨脹取得確定背景
2. Distance Transform → 峰值為確定前景（細胞核心）
3. 未確定區域 = 背景 - 前景（相鄰邊界）
4. Watershed 標記邊界（-1）

In [ ]:
# §5 Watershed 重疊細胞切分

def watershed_split(original, binary):
    '''
    使用 Distance Transform + Watershed 分離相鄰或重疊細胞
    回傳：(markers, dist_transform)
    '''
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    # 確定背景：膨脹後的二值遮罩
    sure_bg = cv2.dilate(binary, kernel, iterations=3)

    # Distance Transform：每個前景像素到最近背景的距離
    dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)

    # 確定前景：距離峰值 50% 以上的區域為細胞核心
    if dist.max() == 0:
        return np.ones_like(binary, dtype=np.int32), dist
    _, sure_fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)

    # 未確定區域（相鄰邊界）
    unknown = cv2.subtract(sure_bg, sure_fg)

    # 連通域標記（每個細胞核心一個唯一 ID）
    _, markers = cv2.connectedComponents(sure_fg)
    markers += 1              # 背景從 1 開始，保留 0 給未確定區域
    markers[unknown == 255] = 0

    # Watershed：邊界處標為 -1
    markers = cv2.watershed(original, markers)

    return markers, dist


print('✅ Watershed 載入完成')

## §6 總控 Pipeline

In [ ]:
# §6 總控 Pipeline：整合前處理 → Watershed → 三種獨立偵測 → 視覺化

def run_pipeline(img, params, detection_method='circular'):
    '''
    完整處理流程：
    preprocess → watershed_split（視覺化用）→ detect_wbc → detect_rbc → detect_platelets → overlay
    RBC / WBC / Platelet 各自採用最適偵測策略，不再依賴統一 Watershed 分類。
    Watershed 結果保留用於 §5 視覺化分頁展示。
    '''
    # 前處理（視覺化各階段）
    stages = preprocess(img, params)

    # Watershed（保留供視覺化展示，不作為計數依據）
    labels, dist = watershed_split(img, stages['morphed'])
    ws_binary = np.zeros_like(stages['morphed'])
    ws_binary[labels > 1] = 255

    # 三種獨立偵測策略，可切換 before / circular
    config = DetectionConfig(method=detection_method)
    wbc_result = detect_wbc(img)
    if detection_method == 'circular':
        rbc_result = detect_rbc_circular_rule(img, config)
        platelet_result = detect_platelets_circular_rule(img, wbc_result, rbc_result, config)
    else:
        rbc_result = detect_rbc(img)
        platelet_result = detect_platelets(img, wbc_result)

    # 繪製標註圖
    annotated = img.copy()
    # WBC：綠色凸包輪廓（full=實綠，partial=橙色）
    if wbc_result['selected'] is not None and \
       wbc_result['status'] in ('full_wbc', 'partial_wbc_border'):
        color = (0, 255, 0) if wbc_result['status'] == 'full_wbc' else (0, 165, 255)
        cv2.drawContours(annotated, [wbc_result['selected']['hull']], -1, color, 2)
    # RBC：紅色圓形
    for c in rbc_result['circles']:
        cv2.circle(annotated, (c['x'], c['y']), c['r'], (0, 0, 255), 2)
    # Platelet：藍色邊框
    for p in platelet_result['platelets']:
        x, y, w, h = p['bbox']
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (255, 0, 0), 2)

    counts = {
        'rbc':      len(rbc_result['circles']),
        'wbc':      1 if wbc_result['status'] in ('full_wbc', 'partial_wbc_border') else 0,
        'platelet': len(platelet_result['platelets']),
    }

    return {
        'stages':          stages,
        'counts':          counts,
        'wbc_result':      wbc_result,
        'rbc_result':      rbc_result,
        'platelet_result': platelet_result,
        'overlay':         annotated,
        'labels':          labels,
        'dist':            dist,
        'ws_binary':       ws_binary,
    }


# 快速測試（使用第一張範例圖）
sample_img = cv2.imread('/content/BCCD_Dataset/BCCD/JPEGImages/BloodImage_00001.jpg')
if sample_img is not None:
    result = run_pipeline(sample_img, PreprocessParams())
    print('偵測結果：', result['counts'])
    show('標註結果（紅:RBC / 綠:WBC / 藍:Platelet）', result['overlay'])
else:
    print('⚠️  請先執行 §0 Setup')

## §7 固定參數評估（5 張圖通用性驗證）

驗證「5 張圖通用」：固定 `PreprocessParams()` 預設值，不手動調整，對 5 張不同圖片評估誤差。

In [ ]:
# §7 評估函式

def parse_gt(xml_path):
    '''解析 BCCD Dataset XML annotation，取得各類細胞真實數量'''
    tree = ET.parse(xml_path)
    root = tree.getroot()
    counts = {'rbc': 0, 'wbc': 0, 'platelet': 0}
    label_map = {'RBC': 'rbc', 'WBC': 'wbc', 'Platelets': 'platelet'}
    for obj in root.findall('object'):
        name = obj.find('name').text
        key = label_map.get(name)
        if key:
            counts[key] += 1
    return counts


def eval_table(pipeline_fn, params, test_images):
    '''
    對多張測試圖執行固定參數 pipeline，輸出預測 vs Ground Truth 誤差表
    test_images：list of (img_path, xml_path)
    '''
    rows = []
    for img_path, xml_path in test_images:
        img = cv2.imread(img_path)
        if img is None:
            print(f'⚠️  無法讀取: {img_path}')
            continue
        result = pipeline_fn(img, params)
        pred = result['counts']
        gt   = parse_gt(xml_path)

        def err(p, g):
            return round(abs(p - g) / g * 100, 1) if g > 0 else (0.0 if p == 0 else 100.0)

        rows.append({
            'image':       os.path.basename(img_path),
            'rbc_pred':    pred['rbc'],      'rbc_gt':  gt['rbc'],      'rbc_err%':  err(pred['rbc'],      gt['rbc']),
            'wbc_pred':    pred['wbc'],      'wbc_gt':  gt['wbc'],      'wbc_err%':  err(pred['wbc'],      gt['wbc']),
            'plt_pred':    pred['platelet'], 'plt_gt':  gt['platelet'], 'plt_err%':  err(pred['platelet'], gt['platelet']),
        })
    return pd.DataFrame(rows)


print('✅ Evaluation 載入完成')

In [ ]:
# §7 執行 5 張圖評估
BCCD_IMG = '/content/BCCD_Dataset/BCCD/JPEGImages'
BCCD_ANN = '/content/BCCD_Dataset/BCCD/Annotations'

test_names = [
    'BloodImage_00001', 'BloodImage_00002', 'BloodImage_00003',
    'BloodImage_00004', 'BloodImage_00005',
]
test_images = [
    (f'{BCCD_IMG}/{n}.jpg', f'{BCCD_ANN}/{n}.xml') for n in test_names
]

# 固定預設參數，不手動調整
fixed_params = PreprocessParams()

df = eval_table(run_pipeline, fixed_params, test_images)
print('=== 固定參數評估結果 ===')
print(df.to_string(index=False))

# 印出各類平均誤差
print()
print(f'RBC 平均誤差：    {df["rbc_err%"].mean():.1f}%')
print(f'WBC 平均誤差：    {df["wbc_err%"].mean():.1f}%')
print(f'Platelet 平均誤差：{df["plt_err%"].mean():.1f}%')

## §8 Streamlit 互動式 App

**流程**：
1. 執行 §8a：將 `app.py` 寫入 Colab 環境
2. 執行 §8b：用 cloudflared 建立公開 URL（不需要帳號）

In [ ]:
%%writefile app.py
# app.py — 血液抹片細胞自動辨識 Streamlit App
import cv2
import numpy as np
import streamlit as st
import os
import glob
from dataclasses import dataclass


@dataclass
class PreprocessParams:
    gaussian_ksize: int = 5
    adaptive_block: int = 71
    adaptive_c:     int = 2
    morph_ksize:    int = 5
    morph_iter:     int = 1


def make_kernel(size):
    size = int(size)
    if size % 2 == 0:
        size += 1
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))


@dataclass
class DetectionConfig:
    method: str = 'circular'
    rbc_sensitive_param2: float = 31
    rbc_strict_param2: float = 33
    rbc_over_count_guard: int = 20
    platelet_rbc_exclusion_pad: int = -6
    platelet_min_area: float = 70
    platelet_max_area: float = 600
    platelet_min_circularity: float = 0.45


def preprocess(img, params):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    blurred = cv2.GaussianBlur(gray, (params.gaussian_ksize, params.gaussian_ksize), 0)
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, params.adaptive_block, params.adaptive_c)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (params.morph_ksize, params.morph_ksize))
    morphed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=params.morph_iter)
    cnts, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(morphed)
    cv2.drawContours(filled, cnts, -1, 255, cv2.FILLED)
    return {'original': img, 'gray': gray, 'enhanced': enhanced,
            'blurred': blurred, 'thresh': thresh, 'morphed': filled}


def watershed_split(original, binary):
    kernel = make_kernel(3)
    sure_bg = cv2.dilate(binary, kernel, iterations=3)
    dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    if dist.max() == 0:
        return np.ones_like(binary, dtype=np.int32), dist
    _, sure_fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    sure_fg = sure_fg.astype(np.uint8)
    unknown = cv2.subtract(sure_bg, sure_fg)
    _, markers = cv2.connectedComponents(sure_fg)
    markers += 1
    markers[unknown == 255] = 0
    return cv2.watershed(original, markers), dist


def detect_wbc(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    purple = cv2.inRange(hsv, np.array([100, 60, 50]), np.array([170, 255, 210]))
    purple = cv2.morphologyEx(purple, cv2.MORPH_OPEN, make_kernel(3), iterations=1)
    merged = cv2.dilate(purple, make_kernel(15), iterations=1)
    merged = cv2.morphologyEx(merged, cv2.MORPH_CLOSE, make_kernel(15), iterations=1)
    img_h, img_w = merged.shape
    contours, _ = cv2.findContours(merged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    raw_candidates = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 500: continue
        x, y, w, h = cv2.boundingRect(cnt)
        touches = x <= 1 or y <= 1 or x + w >= img_w - 1 or y + h >= img_h - 1
        m = cv2.moments(cnt)
        cx = m['m10'] / m['m00'] if m['m00'] else x + w / 2
        cy = m['m01'] / m['m00'] if m['m00'] else y + h / 2
        raw_candidates.append({'contour': cnt, 'area': float(area),
                                'bbox': (x, y, w, h), 'touches': touches, 'center': (cx, cy)})
    visited = [False] * len(raw_candidates)
    clusters = []
    for i in range(len(raw_candidates)):
        if visited[i]: continue
        stack, cluster = [i], []
        visited[i] = True
        while stack:
            cur = stack.pop(); cluster.append(cur)
            x1, y1 = raw_candidates[cur]['center']
            for j in range(len(raw_candidates)):
                if not visited[j]:
                    x2, y2 = raw_candidates[j]['center']
                    if ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5 < 140:
                        visited[j] = True; stack.append(j)
        clusters.append(cluster)
    hulls = []
    for cluster in clusters:
        points = np.vstack([raw_candidates[i]['contour'] for i in cluster])
        hull = cv2.convexHull(points)
        area = cv2.contourArea(hull)
        x, y, w, h = cv2.boundingRect(hull)
        touches = x <= 1 or y <= 1 or x + w >= img_w - 1 or y + h >= img_h - 1
        hulls.append({'hull': hull, 'area': float(area), 'bbox': (x, y, w, h), 'touches': touches})
    non_border = [h for h in hulls if not h['touches'] and h['area'] >= 6000]
    border     = [h for h in hulls if     h['touches'] and h['area'] >= 6000]
    if non_border:   selected, status = max(non_border, key=lambda h: h['area']), 'full_wbc'
    elif border:     selected, status = max(border,     key=lambda h: h['area']), 'partial_wbc_border'
    elif hulls:      selected, status = max(hulls,      key=lambda h: h['area']), 'no_confident_wbc'
    else:            selected, status = None, 'no_candidate'
    return {'purple_mask': purple, 'merged_mask': merged,
            'hulls': hulls, 'selected': selected, 'status': status}


def detect_rbc(bgr, param2=32):
    # param2=32：由 YOLO 輔助診斷 + GT XML param2 掃描決定（平均誤差 28.1%）
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    blurred = cv2.medianBlur(enhanced, 5)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1.2,
                                minDist=35, param1=50, param2=param2, minRadius=18, maxRadius=55)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    h, s, v = cv2.split(hsv); L, A, B = cv2.split(lab)
    purple = cv2.inRange(hsv, np.array([100, 60, 50]), np.array([170, 255, 210]))
    wbc_exclusion = cv2.dilate(purple, make_kernel(25), iterations=1)
    mask_hsv = (((h >= 150) | (h <= 10)) & (s >= 8) & (s <= 100) &
                (v >= 100) & (v <= 240)).astype(np.uint8) * 255
    mask_lab = ((A >= 132) & (B >= 118) & (L >= 110) & (L <= 225)).astype(np.uint8) * 255
    rbc_mask = cv2.bitwise_or(mask_hsv, mask_lab)
    rbc_mask[wbc_exclusion > 0] = 0
    rbc_mask = cv2.morphologyEx(rbc_mask, cv2.MORPH_OPEN, make_kernel(3), iterations=1)
    accepted = []
    if circles is not None:
        circles_int = np.round(circles[0]).astype(int)
        img_h, img_w = gray.shape
        for x, y, r in circles_int:
            if x < 0 or y < 0 or x >= img_w or y >= img_h: continue
            cm = np.zeros(gray.shape, np.uint8); cv2.circle(cm, (x, y), r, 255, -1)
            area = np.sum(cm > 0)
            rbc_ratio = np.sum((rbc_mask > 0) & (cm > 0)) / (area + 1e-6)
            wbc_ratio = np.sum((wbc_exclusion > 0) & (cm > 0)) / (area + 1e-6)
            if wbc_ratio > 0.08 or rbc_ratio < 0.15: continue
            accepted.append({'x': int(x), 'y': int(y), 'r': int(r)})
    return {'rbc_mask': rbc_mask, 'wbc_exclusion': wbc_exclusion, 'circles': accepted}


def detect_rbc_circular_rule(bgr, config=None):
    # HoughCircles 產生圓形候選，再補上 area/circularity 作為分類依據。
    # 候選過多時用固定規則切換到較嚴格門檻，避免單張手動調參。
    if config is None:
        config = DetectionConfig()

    sensitive = detect_rbc(bgr, param2=config.rbc_sensitive_param2)
    if len(sensitive['circles']) > config.rbc_over_count_guard:
        result = detect_rbc(bgr, param2=config.rbc_strict_param2)
    else:
        result = sensitive

    for c in result['circles']:
        c['area'] = float(np.pi * c['r'] * c['r'])
        c['circularity'] = 1.0
        c['bbox'] = (int(c['x'] - c['r']), int(c['y'] - c['r']), int(c['r'] * 2), int(c['r'] * 2))

    result['method'] = 'circular_rule'
    return result


def detect_platelets(bgr, wbc_result):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    raw_mask = cv2.inRange(hsv, np.array([110, 80, 30]), np.array([160, 255, 210]))
    raw_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN, make_kernel(3), iterations=1)
    exclude = np.zeros(raw_mask.shape, np.uint8)
    if wbc_result['selected'] is not None:
        cv2.drawContours(exclude, [wbc_result['selected']['hull']], -1, 255, -1)
        exclude = cv2.dilate(exclude, make_kernel(35), iterations=1)
    pm = raw_mask.copy(); pm[exclude > 0] = 0
    pm = cv2.morphologyEx(pm, cv2.MORPH_OPEN, make_kernel(3), iterations=1)
    cnts, _ = cv2.findContours(pm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_h, img_w = pm.shape
    platelets = []
    for cnt in cnts:
        area = cv2.contourArea(cnt)
        if not (8 <= area <= 250): continue
        x, y, w, h = cv2.boundingRect(cnt)
        if x <= 1 or y <= 1 or x + w >= img_w - 1 or y + h >= img_h - 1: continue
        aspect = w / h if h else 0
        if aspect < 0.35 or aspect > 2.8: continue
        perimeter = cv2.arcLength(cnt, True)
        circ = 4 * np.pi * area / (perimeter * perimeter) if perimeter else 0
        if circ < 0.2: continue
        m = cv2.moments(cnt)
        cx = int(m['m10'] / m['m00']) if m['m00'] else x + w // 2
        cy = int(m['m01'] / m['m00']) if m['m00'] else y + h // 2
        platelets.append({'contour': cnt, 'area': float(area),
                           'bbox': (x, y, w, h), 'center': (cx, cy)})
    return {'platelet_mask': pm, 'platelets': platelets}


def detect_platelets_circular_rule(bgr, wbc_result, rbc_result, config=None):
    # HSV/Lab 淡藍紫色遮罩 → WBC 排除 → RBC 圓形區排除 → 面積/圓形度/長寬比篩選。
    if config is None:
        config = DetectionConfig()

    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    h, s, v = cv2.split(hsv)
    L, A, B = cv2.split(lab)

    platelet_mask = (((h >= 105) & (h <= 130) &
                      (s >= 25)  & (s <= 95)  &
                      (v >= 150) & (v <= 225) &
                      (A >= 130) & (A <= 138) &
                      (B >= 108) & (B <= 116) &
                      (L >= 145) & (L <= 205))).astype(np.uint8) * 255

    platelet_mask = cv2.morphologyEx(platelet_mask, cv2.MORPH_OPEN, make_kernel(3), iterations=1)

    exclude = np.zeros(platelet_mask.shape, np.uint8)
    if wbc_result['selected'] is not None:
        cv2.drawContours(exclude, [wbc_result['selected']['hull']], -1, 255, -1)
        exclude = cv2.dilate(exclude, make_kernel(35), iterations=1)

    for c in rbc_result['circles']:
        radius = max(1, int(c['r'] + config.platelet_rbc_exclusion_pad))
        cv2.circle(exclude, (c['x'], c['y']), radius, 255, -1)

    platelet_mask[exclude > 0] = 0
    platelet_mask = cv2.morphologyEx(platelet_mask, cv2.MORPH_CLOSE, make_kernel(5), iterations=1)

    contours, _ = cv2.findContours(platelet_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_h, img_w = platelet_mask.shape
    platelets = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (config.platelet_min_area <= area <= config.platelet_max_area):
            continue

        x, y, w, h_box = cv2.boundingRect(cnt)
        if x <= 5 or y <= 5 or x + w >= img_w - 5 or y + h_box >= img_h - 5:
            continue

        aspect = w / h_box if h_box else 0
        if aspect < 0.5 or aspect > 2.0:
            continue

        perimeter = cv2.arcLength(cnt, True)
        circularity = 4 * np.pi * area / (perimeter * perimeter) if perimeter else 0
        if circularity < config.platelet_min_circularity:
            continue

        m = cv2.moments(cnt)
        cx = int(m['m10'] / m['m00']) if m['m00'] else x + w // 2
        cy = int(m['m01'] / m['m00']) if m['m00'] else y + h_box // 2
        platelets.append({'contour': cnt, 'area': float(area),
                           'bbox': (x, y, w, h_box), 'center': (cx, cy),
                           'circularity': float(circularity), 'aspect': float(aspect)})

    return {'raw_mask': platelet_mask, 'exclude_mask': exclude,
            'platelet_mask': platelet_mask, 'platelets': platelets,
            'method': 'circular_rule'}


def run_pipeline(img_bytes, gk, ab, ac, mk, detection_method='circular'):
    img = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
    params = PreprocessParams(gaussian_ksize=gk, adaptive_block=ab, adaptive_c=ac, morph_ksize=mk)
    stages = preprocess(img, params)
    labels, dist = watershed_split(img, stages['morphed'])
    config = DetectionConfig(method=detection_method)
    wbc_result = detect_wbc(img)
    if detection_method == 'circular':
        rbc_result = detect_rbc_circular_rule(img, config)
        platelet_result = detect_platelets_circular_rule(img, wbc_result, rbc_result, config)
    else:
        rbc_result = detect_rbc(img)
        platelet_result = detect_platelets(img, wbc_result)
    annotated = img.copy()
    if wbc_result['selected'] is not None and \
       wbc_result['status'] in ('full_wbc', 'partial_wbc_border'):
        color = (0, 255, 0) if wbc_result['status'] == 'full_wbc' else (0, 165, 255)
        cv2.drawContours(annotated, [wbc_result['selected']['hull']], -1, color, 2)
    for c in rbc_result['circles']:
        cv2.circle(annotated, (c['x'], c['y']), c['r'], (0, 0, 255), 2)
    for p in platelet_result['platelets']:
        x, y, w, h = p['bbox']
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (255, 0, 0), 2)
    counts = {
        'rbc':      len(rbc_result['circles']),
        'wbc':      1 if wbc_result['status'] in ('full_wbc', 'partial_wbc_border') else 0,
        'platelet': len(platelet_result['platelets']),
    }
    return {'img': img, 'stages': stages, 'counts': counts,
            'wbc_result': wbc_result, 'rbc_result': rbc_result,
            'platelet_result': platelet_result, 'overlay': annotated,
            'labels': labels, 'dist': dist}


def to_rgb(img):
    return img if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# ── Streamlit UI ─────────────────────────────────────────────────────────────
st.set_page_config(page_title='血液抹片分析', layout='wide', page_icon='🔬')
st.title('🔬 血液抹片細胞自動辨識與計數系統')
st.caption('影像處理 114-2 ｜ OpenCV · CLAHE · Adaptive Thresholding · Hough · Watershed · Streamlit')

DEFAULTS = {'gk': 5, 'ab': 71, 'ac': 2, 'mk': 5}
for k, v in DEFAULTS.items():
    if k not in st.session_state:
        st.session_state[k] = v

with st.sidebar:
    st.header('⚙️ 設定')
    src_mode = st.radio('影像來源', ['上傳圖片', '範例圖片'])
    img_bytes = None
    if src_mode == '上傳圖片':
        uploaded = st.file_uploader('選擇血液抹片影像', type=['jpg', 'jpeg', 'png'])
        if uploaded:
            img_bytes = uploaded.read()
    else:
        sample_dir = '/content/BCCD_Dataset/BCCD/JPEGImages'
        sample_names = [
            'BloodImage_00001.jpg', 'BloodImage_00002.jpg', 'BloodImage_00003.jpg',
            'BloodImage_00004.jpg', 'BloodImage_00005.jpg',
        ]
        samples = [os.path.join(sample_dir, name) for name in sample_names
                   if os.path.exists(os.path.join(sample_dir, name))]
        if samples:
            idx = st.selectbox('選擇範例圖片', range(len(samples)),
                               format_func=lambda i: os.path.basename(samples[i]))
            with open(samples[idx], 'rb') as f:
                img_bytes = f.read()
        else:
            st.warning('找不到 BCCD dataset 或指定的 00001–00005 範例圖，請先執行 §0 Setup cell')
    st.divider()
    st.subheader('偵測模式')
    detection_method = st.selectbox('分類方法', ['circular', 'before'],
                                    format_func=lambda x: '面積 + 圓形度規則' if x == 'circular' else '原本混合方法')
    st.divider()
    st.subheader('📊 前處理參數')
    if st.button('🔄 Reset 預設值'):
        for k, v in DEFAULTS.items():
            st.session_state[k] = v
        st.rerun()
    gk = st.slider('Gaussian Kernel Size', 3, 11, step=2, key='gk')
    ab = st.slider('Adaptive Block Size', 11, 101, step=2, key='ab')
    ac = st.slider('Adaptive C', 1, 15, key='ac')
    mk = st.slider('Morph Kernel Size', 1, 9, step=2, key='mk')

if img_bytes is None:
    st.info('👈 請在左側選擇或上傳一張血液抹片影像')
    st.stop()

with st.spinner('處理中...'):
    result = run_pipeline(img_bytes, st.session_state['gk'], st.session_state['ab'],
                          st.session_state['ac'], st.session_state['mk'], detection_method)

tab1, tab2, tab3 = st.tabs(['🔬 Pipeline 可視化', '🧬 分類結果', '💧 Watershed'])

with tab1:
    st.subheader('影像前處理各階段')
    stages = result['stages']
    stage_items = [('original','① 原圖'),('gray','② 灰階'),('enhanced','③ CLAHE 增強'),
                   ('blurred','④ Gaussian 模糊'),('thresh','⑤ Adaptive Thresholding'),
                   ('morphed','⑥ Morph Closing + 輪廓填充')]
    cols = st.columns(3)
    for i, (key, label) in enumerate(stage_items):
        with cols[i % 3]:
            st.caption(label)
            st.image(to_rgb(stages[key]), use_container_width=True)
    st.divider()
    st.caption('RBC 顏色驗證遮罩（白色 = 通過驗證的粉紅色區域）')
    st.image(result['rbc_result']['rbc_mask'], use_container_width=True)

with tab2:
    st.subheader('細胞分類結果')
    col_img, col_stat = st.columns([2, 1])
    with col_img:
        st.image(to_rgb(result['overlay']),
                 caption='標註圖（🔴 RBC圓形 ／ 🟢 WBC凸包 ／ 🔵 Platelet邊框）',
                 use_container_width=True)
    with col_stat:
        counts = result['counts']
        st.metric('🔴 RBC（紅血球）',     counts['rbc'])
        st.metric('🟢 WBC（白血球）',     counts['wbc'])
        st.metric('🔵 Platelet（血小板）', counts['platelet'])
        st.divider()
        st.metric('總計', sum(counts.values()))
        st.caption(f'WBC 狀態：{result["wbc_result"]["status"]}')
    st.divider()
    _, buf = cv2.imencode('.png', result['overlay'])
    st.download_button('⬇️ 下載標註圖 PNG', buf.tobytes(), 'annotated.png', 'image/png')

with tab3:
    st.subheader('Watershed 重疊細胞切分流程')

    binary = result['stages']['morphed']
    dist = result['dist']
    labels = result['labels']

    kernel = make_kernel(3)
    sure_bg = cv2.dilate(binary, kernel, iterations=3)
    if dist.max() > 0:
        _, sure_fg = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
        sure_fg = sure_fg.astype(np.uint8)
    else:
        sure_fg = np.zeros_like(binary)
    unknown = cv2.subtract(sure_bg, sure_fg)

    dist_norm = cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    dist_heat = cv2.applyColorMap(dist_norm, cv2.COLORMAP_JET)

    boundary = result['img'].copy()
    boundary[labels == -1] = [0, 0, 255]

    marker_vis = np.zeros_like(result['img'])
    marker_ids = labels.copy()
    marker_ids[marker_ids < 0] = 0
    if marker_ids.max() > 0:
        marker_norm = cv2.normalize(marker_ids.astype(np.float32), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        marker_vis = cv2.applyColorMap(marker_norm, cv2.COLORMAP_TURBO)
        marker_vis[labels == -1] = [0, 0, 255]

    ws_regions = int(len(np.unique(labels[labels > 1])))
    boundary_pixels = int(np.sum(labels == -1))

    m1, m2 = st.columns(2)
    m1.metric('Watershed regions', ws_regions)
    m2.metric('Boundary pixels', boundary_pixels)

    col1, col2, col3 = st.columns(3)
    with col1:
        st.caption('① Binary mask（Morph Closing + 輪廓填充）')
        st.image(binary, use_container_width=True)
    with col2:
        st.caption('② Distance Transform（暖色 = 細胞核心）')
        st.image(to_rgb(dist_heat), use_container_width=True)
    with col3:
        st.caption('③ Sure foreground markers')
        st.image(sure_fg, use_container_width=True)

    col4, col5, col6 = st.columns(3)
    with col4:
        st.caption('④ Unknown border zone')
        st.image(unknown, use_container_width=True)
    with col5:
        st.caption('⑤ Marker labels（紅色 = 分水嶺邊界）')
        st.image(to_rgb(marker_vis), use_container_width=True)
    with col6:
        st.caption('⑥ Boundary overlay（紅線 = Watershed 切分線）')
        st.image(to_rgb(boundary), use_container_width=True)

In [ ]:
# §8b 啟動 Streamlit App（Colab 環境，使用 cloudflared）
import subprocess
import time
import re

# 啟動 Streamlit（背景執行）
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

# 啟動 cloudflared tunnel，從 stderr 讀取公開 URL
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

# 等待並解析 trycloudflare.com URL
url = None
for line in cf_proc.stderr:
    text = line.decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', text)
    if match:
        url = match.group(0)
        break

print(f'✅ Streamlit App 已啟動')
print(f'🌐 公開網址：{url}')
print('點擊連結即可開啟互動式 App（3 分鐘 Demo 用這個 URL）')